# Demo 1 — Một model tốt vẫn chết sau một năm

**Slide:** 2 và 5 · **Thời lượng:** 30–60 giây · **Hình:** `01_model_decay.png`

## Ý chính

Model dự báo nhu cầu hàng hoá được train 12 tháng đầu và chạy tốt. Nhưng thế giới thay đổi:

- Đối thủ mới làm cầu giảm dần.
- Promotion ngày càng hiệu quả hơn.

Cùng một model, MAE tăng gần **4 lần** sau một năm. Retrain ở tháng 18 kéo MAE xuống.

> **Câu chốt:** deploy là bắt đầu, không phải kết thúc.

## Bước 0 — Thiết lập

In [ ]:
# --- Thiết lập: tìm và import helper từ ../code-demo ---
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

for candidate in [
    Path.cwd() / "code-demo",
    Path.cwd().parent / "code-demo",
    Path.cwd().parent.parent / "code-demo",
]:
    if (candidate / "common.py").exists():
        CODE_DIR = candidate.resolve()
        break
else:
    raise FileNotFoundError("Không tìm thấy code-demo/common.py")

sys.path.insert(0, str(CODE_DIR))

from common import save, setup  # noqa: E402

setup()
print("code-demo:", CODE_DIR)
print("Hình sẽ lưu vào:", CODE_DIR / "figures")

## Bước 1 — Mô phỏng 24 tháng dữ liệu

- 12 tháng đầu: thế giới ổn định.
- Từ tháng 10: drift bắt đầu.
- Nhu cầu = trend + mùa vụ + promo + drift + nhiễu.

In [ ]:
rng = np.random.default_rng(42)

months = np.arange(24)
promo = rng.integers(0, 2, len(months))
season = 15 * np.sin(2 * np.pi * months / 12)

# Sau tháng 10: đối thủ mới làm cầu giảm dần; promo ngày càng hiệu quả.
drift = np.where(months >= 10, -3.0 * (months - 9), 0.0)
promo_effect = 8.0 + np.where(months >= 10, 1.5 * (months - 9), 0.0)

demand = 100 + 1.5 * months + season + promo_effect * promo + drift
demand = demand + rng.normal(0, 3, len(months))


def make_features(months, promo):
    return np.c_[
        months,
        np.sin(2 * np.pi * months / 12),
        np.cos(2 * np.pi * months / 12),
        promo,
    ]


X = make_features(months, promo)
print("Số tháng:", len(months))
print("5 giá trị nhu cầu đầu:", np.round(demand[:5], 1))

## Bước 2 — Train lần 1, deploy, rồi retrain ở tháng 18

- Model 1: train 12 tháng đầu (0–11).
- Model 2: retrain bằng 12 tháng gần nhất (6–17).
- Đo MAE trên 6 tháng đầu sau deploy và 6 tháng cuối.

In [ ]:
def fit_linear(X, y):
    Xb = np.c_[np.ones(len(X)), X]
    w, *_ = np.linalg.lstsq(Xb, y, rcond=None)
    return w


def predict_linear(X, w):
    return np.c_[np.ones(len(X)), X] @ w


w1 = fit_linear(X[:12], demand[:12])
pred1 = predict_linear(X, w1)

w2 = fit_linear(X[6:18], demand[6:18])
pred2 = predict_linear(X, w2)

mae1 = np.abs(pred1 - demand)
mae2 = np.abs(pred2 - demand)

print("=== DEMO 1: model suy giảm theo thời gian ===")
print(f"MAE 6 tháng đầu sau deploy : {mae1[12:18].mean():6.2f}")
print(f"MAE 6 tháng cuối           : {mae1[18:24].mean():6.2f}  ← tăng {mae1[18:24].mean() / mae1[12:18].mean():.1f}×")
print(f"MAE sau retrain (tháng 18) : {mae2[18:24].mean():6.2f}")
print("Bài học: deploy là bắt đầu; drift làm sai số tăng, retrain kéo lại.")

## Bước 3 — Vẽ hình cho slide

Hình gồm 2 phần:

- Trên: nhu cầu thực tế, dự báo của model cũ, dự báo sau retrain.
- Dưới: MAE theo tháng — cột đỏ là model cũ, đường xanh là sau retrain.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True, height_ratios=[2, 1])

ax1.plot(months, demand, "o-", label="Nhu cầu thực tế", color="#1f77b4")
ax1.plot(months, pred1, "--", label="Dự báo (model train 12 tháng đầu)", color="#d62728")
ax1.plot(months[6:], pred2[6:], "-.", label="Dự báo (retrain tháng 18)", color="#2ca02c")
ax1.axvline(12, color="gray", ls=":", lw=1.5)
ax1.text(12.2, ax1.get_ylim()[1] * 0.95, "deploy", color="gray", fontsize=9)
ax1.axvline(18, color="gray", ls=":", lw=1.5)
ax1.text(18.2, ax1.get_ylim()[1] * 0.95, "retrain", color="gray", fontsize=9)
ax1.set_ylabel("Nhu cầu")
ax1.set_title("Model tốt lúc bàn giao vẫn hỏng dần vì thế giới thay đổi")
ax1.legend(loc="upper left", fontsize=9)

ax2.bar(months, mae1, color="#d62728", alpha=0.6, label="MAE model cũ")
ax2.plot(months[18:], mae2[18:], "o-", color="#2ca02c", label="MAE sau retrain")
ax2.set_xlabel("Tháng")
ax2.set_ylabel("MAE")
ax2.legend(fontsize=9)

save(fig, "01_model_decay.png")
plt.show()

## Đọc kết quả & ý nghĩa cho slide

| Chỉ số | Giá trị kỳ vọng |
|---|---|
| MAE 6 tháng đầu sau deploy | **~4.9** |
| MAE 6 tháng cuối | **~18.8** |
| Mức tăng | **~3.8×** |
| MAE sau retrain tháng 18 | **~8.4** |

**Nói trên slide:**

> “Model train 12 tháng đầu, MAE khoảng 4,9. Một năm sau, cùng model đó, MAE lên 18,8 — gần gấp 4. Model không đổi; thế giới đổi. Retrain ở tháng 18 kéo MAE về 8,4.”

**Câu chốt:** deploy là bắt đầu, không phải kết thúc.

**Dùng cho slide nào?**

- Slide 2 — “Một model tốt có thể chết sau một năm”.
- Slide 5 — train-serving skew / thế giới thay đổi theo thời gian.